# Assignment 4.1 — Model Store: Model Group, Model Package, and Model Card

**Course:** AAI-540 · **Author:** Sourangshu Pal (Group 1)

This notebook implements a **Model Store** for the binary breast-cancer classifier built in Lab 4.1. Section 1 condenses the Lab 4.1 pipeline (train XGBoost → batch transform → deploy endpoint → invoke → clean up), and Sections 2–4 add the three graded artifacts to the SageMaker Model Registry:

| Part | boto3 call | Verification (📸 screenshot) |
|---|---|---|
| 1. Model Package Group | `create_model_package_group` | `describe_model_package_group` |
| 2. Model Package | `create_model_package` | `describe_model_package` |
| 3. Model Card | `create_model_card` | `describe_model_card` |

**Run order:** Run cells top to bottom in SageMaker Studio. Screenshot the output of each cell marked 📸 — those three screenshots plus this notebook become `Sourangshu_Pal_Assignment4.1.pdf`.

---
## Section 1 — Train and deploy the Lab 4.1 model (condensed from Lab 4.1)

The cells below mirror Lab 4.1 (`01-train-and-deploy.ipynb`): same data (Wisconsin Diagnostic Breast Cancer, 569 × 32), same 80/10/10 split, same XGBoost 1.7-1 training configuration, same batch-transform and endpoint steps. Exploratory cells from the lab are summarized here rather than repeated. The variables needed by the Model Store sections (`job_name`, `image`, `model_data`, `role`, `region`) are produced in this section.

### 1.1 Setup

In [ ]:
!pip3 install -U sagemaker

import os
import boto3
import sagemaker

role = sagemaker.get_execution_role()
sess = sagemaker.Session()
region = sess.boto_region_name

bucket = sess.default_bucket()
prefix = "DEMO-breast-cancer-prediction-xgboost-highlevel"
print(f"region={region}  bucket={bucket}")

### 1.2 Load the Wisconsin Diagnostic Breast Cancer dataset

569 observations × 32 columns from the public `sagemaker-example-files-prod-{region}` bucket. `diagnosis` (M/B) is encoded to 1/0.

In [ ]:
import pandas as pd
import numpy as np

s3 = boto3.client("s3")

filename = "wdbc.csv"
s3.download_file(
    f"sagemaker-example-files-prod-{region}", "datasets/tabular/breast_cancer/wdbc.csv", filename
)
data = pd.read_csv(filename, header=None)

# specify columns extracted from wbdc.names
data.columns = [
    "id",
    "diagnosis",
    "radius_mean",
    "texture_mean",
    "perimeter_mean",
    "area_mean",
    "smoothness_mean",
    "compactness_mean",
    "concavity_mean",
    "concave points_mean",
    "symmetry_mean",
    "fractal_dimension_mean",
    "radius_se",
    "texture_se",
    "perimeter_se",
    "area_se",
    "smoothness_se",
    "compactness_se",
    "concavity_se",
    "concave points_se",
    "symmetry_se",
    "fractal_dimension_se",
    "radius_worst",
    "texture_worst",
    "perimeter_worst",
    "area_worst",
    "smoothness_worst",
    "compactness_worst",
    "concavity_worst",
    "concave points_worst",
    "symmetry_worst",
    "fractal_dimension_worst",
]

data["diagnosis"] = data["diagnosis"].apply(lambda x: ((x == "M")) + 0)
data.sample(8)

### 1.3 Split 80/10/10 (train / validation / batch) and upload to S3

In [ ]:
rand_split = np.random.rand(len(data))
train_list = rand_split < 0.8
val_list = (rand_split >= 0.8) & (rand_split < 0.9)
batch_list = rand_split >= 0.9

data_train = data[train_list].drop(["id"], axis=1)
data_val = data[val_list].drop(["id"], axis=1)
data_batch = data[batch_list].drop(["diagnosis"], axis=1)
data_batch_noID = data_batch.drop(["id"], axis=1)

train_file = "train_data.csv"
data_train.to_csv(train_file, index=False, header=False)
sess.upload_data(train_file, key_prefix="{}/train".format(prefix))

validation_file = "validation_data.csv"
data_val.to_csv(validation_file, index=False, header=False)
sess.upload_data(validation_file, key_prefix="{}/validation".format(prefix))

batch_file = "batch_data.csv"
data_batch.to_csv(batch_file, index=False, header=False)
sess.upload_data(batch_file, key_prefix="{}/batch".format(prefix))

batch_file_noID = "batch_data_noID.csv"
data_batch_noID.to_csv(batch_file_noID, index=False, header=False)
sess.upload_data(batch_file_noID, key_prefix="{}/batch".format(prefix))
print("uploaded train/validation/batch data to s3://{}/{}".format(bucket, prefix))

### 1.4 Train XGBoost (same configuration as Lab 4.1)

`binary:logistic`, max_depth 5, eta 0.2, gamma 4, min_child_weight 6, subsample 0.8, 100 rounds, `ml.m5.xlarge`.

In [ ]:
%%time
from time import gmtime, strftime

job_name = "xgb-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
output_location = "s3://{}/{}/output/{}".format(bucket, prefix, job_name)
image = sagemaker.image_uris.retrieve(
    framework="xgboost", region=boto3.Session().region_name, version="1.7-1"
)

sm_estimator = sagemaker.estimator.Estimator(
    image,
    role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size=50,
    input_mode="File",
    output_path=output_location,
    sagemaker_session=sess,
)

sm_estimator.set_hyperparameters(
    objective="binary:logistic",
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.8,
    verbosity=0,
    num_round=100,
)

train_data = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/train".format(bucket, prefix),
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
validation_data = sagemaker.inputs.TrainingInput(
    "s3://{}/{}/validation".format(bucket, prefix),
    distribution="FullyReplicated",
    content_type="text/csv",
    s3_data_type="S3Prefix",
)
data_channels = {"train": train_data, "validation": validation_data}

sm_estimator.fit(inputs=data_channels, job_name=job_name, logs=True)

### 1.5 Batch transform

First a baseline transform (no filters), then the join configuration: `input_filter="$[1:]"` drops the id column before inference, `join_source="Input"` associates predictions with input records, `output_filter="$[0,-1]"` keeps only id + probability.

In [ ]:
%%time
sm_transformer = sm_estimator.transformer(1, "ml.m5.xlarge")

# baseline transform job (no filters)
input_location = "s3://{}/{}/batch/{}".format(bucket, prefix, batch_file_noID)
sm_transformer.transform(input_location, content_type="text/csv", split_type="Line")
sm_transformer.wait()

In [ ]:
import re


def get_csv_output_from_s3(s3uri, batch_file):
    file_name = "{}.out".format(batch_file)
    match = re.match("s3://([^/]+)/(.*)", "{}/{}".format(s3uri, file_name))
    output_bucket, output_prefix = match.group(1), match.group(2)
    s3.download_file(output_bucket, output_prefix, file_name)
    return pd.read_csv(file_name, sep=",", header=None)


output_df = get_csv_output_from_s3(sm_transformer.output_path, batch_file_noID)
print("baseline batch predictions (probability of malignancy):")
output_df.head(8)

In [ ]:
%%time
# transform with join: keep id alongside the prediction
sm_transformer.transform(
    input_location,
    split_type="Line",
    content_type="text/csv",
    input_filter="$[1:]",
    join_source="Input",
    output_filter="$[0,-1]",
)
sm_transformer.wait()

output_df = get_csv_output_from_s3(sm_transformer.output_path, batch_file)
output_df.columns = ["id", "probability_of_malignancy"]
output_df.head(8)

### 1.6 Register the trained model as a SageMaker Model and capture its artifact

This is the Lab 4.1 `create_model` step. `model_data` (the S3 URI of the trained XGBoost artifact) and `image` are the two values the Model Package in Section 3 documents.

In [ ]:
sagemaker_client = boto3.client("sagemaker")

model_name = job_name  # model named after its training job
print(model_name)

info = sagemaker_client.describe_training_job(TrainingJobName=model_name)
model_data = info["ModelArtifacts"]["S3ModelArtifacts"]
print(f"model artifact: {model_data}")

primary_container = {"Image": image, "ModelDataUrl": model_data}
create_model_response = sagemaker_client.create_model(
    ModelName=model_name, ExecutionRoleArn=role, PrimaryContainer=primary_container
)
print(create_model_response["ModelArn"])

### 1.7 Deploy to a real-time endpoint, invoke once, then delete the endpoint

In [ ]:
from time import gmtime, strftime

# endpoint config
endpoint_config_name = "assignment4-1-endpoint-config-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
instance_type = "ml.m5.xlarge"

endpoint_config_response = sagemaker_client.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "variant1",
            "ModelName": model_name,
            "InstanceType": instance_type,
            "InitialInstanceCount": 1,
        }
    ],
)
print(f"Created EndpointConfig: {endpoint_config_response['EndpointConfigArn']}")

# endpoint
endpoint_name = "assignment4-1-endpoint-" + strftime("%Y-%m-%d-%H-%M-%S", gmtime())
sagemaker_client.create_endpoint(
    EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name
)

from time import sleep

while True:
    res = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)
    state = res["EndpointStatus"]
    if state == "InService":
        print("Endpoint in Service")
        break
    elif state == "Creating":
        print("Endpoint still creating...")
        sleep(60)
    else:
        print("Endpoint Creation Error - Check SageMaker Console")
        break

In [ ]:
sagemaker_runtime = boto3.client("sagemaker-runtime", region_name=region)

response = sagemaker_runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="text/csv",
    Body=data_batch_noID.to_csv(header=None, index=False).strip("\n").split("\n")[0],
)
prediction = response["Body"].read().decode("utf-8")
print(f"sample probability of malignancy: {prediction}")

# cleanup: delete the endpoint (the model, artifacts, and registry entries stay)
sagemaker_client.delete_endpoint(EndpointName=endpoint_name)
print(f"deleted endpoint {endpoint_name}")

---
## Part 1 — Model Package Group (📸 screenshot the describe output)

The Model Group tracks every versioned experiment for this model lineage. Group name follows the assignment's example naming convention.

In [ ]:
MODEL_PACKAGE_GROUP_NAME = "xgboost-breast-cancer-detection"
MODEL_PACKAGE_GROUP_DESCRIPTION = (
    "Versioned lineage of XGBoost binary classifiers predicting malignancy "
    "(probability) from the 30 WDBC cytology features. A new model package is "
    "added whenever the algorithm, input data, features, or hyperparameters change."
)
print(f"description length: {len(MODEL_PACKAGE_GROUP_DESCRIPTION)} chars")

sagemaker_client.create_model_package_group(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    ModelPackageGroupDescription=MODEL_PACKAGE_GROUP_DESCRIPTION,
)
print(f"created model package group: {MODEL_PACKAGE_GROUP_NAME}")

In [ ]:
# 📸 SCREENSHOT 1 (Part 1): describe_model_package_group output
sagemaker_client.describe_model_package_group(ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME)

---
## Part 2 — Model Package (📸 screenshot the describe output)

The Model Package records this specific experiment: the inference container image, the binary model artifact from the training job, supported instance types, content types, and approval status (`PendingManualApproval` — a human reviews before production use).

In [ ]:
model_package_response = sagemaker_client.create_model_package(
    ModelPackageGroupName=MODEL_PACKAGE_GROUP_NAME,
    ModelPackageDescription=(
        "XGBoost 1.7-1 binary:logistic classifier trained on the WDBC 80/10 split "
        "(ml.m5.xlarge, 100 rounds). Artifact from training job "
        f"{job_name}. Deployable to a real-time endpoint or batch transform."
    ),
    InferenceSpecification={
        "Containers": [
            {
                "Image": image,
                "ModelDataUrl": model_data,
                "Framework": "XGBoost",
                "FrameworkVersion": "1.7-1",
            }
        ],
        "SupportedTransformInstanceTypes": ["ml.m5.xlarge"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.xlarge"],
        "SupportedContentTypes": ["text/csv"],
        "SupportedResponseMIMETypes": ["text/csv"],
    },
    ModelApprovalStatus="PendingManualApproval",
)
MODEL_PACKAGE_ARN = model_package_response["ModelPackageArn"]
print(MODEL_PACKAGE_ARN)

In [ ]:
# 📸 SCREENSHOT 2 (Part 2): describe_model_package output
sagemaker_client.describe_model_package(ModelPackageArn=MODEL_PACKAGE_ARN)

---
## Part 3 — Model Card (📸 screenshot the describe output)

The Model Card is the qualitative snapshot: what the model is, how it was trained, its hyperparameters, intended (and out-of-scope) uses, evaluation approach, and maintenance notes — so future maintainers don't have to rediscover why the model was built this way.

In [ ]:
MODEL_CARD_NAME = "xgboost-breast-cancer-detection-model-card"

model_card_content = {
    "model_overview": {
        "model_description": (
            "Gradient-boosted trees (XGBoost) binary classifier that scores the "
            "probability of malignancy for a breast-mass cytology sample from 30 "
            "numeric cell-nucleus features (mean/SE/worst of radius, texture, "
            "perimeter, area, smoothness, compactness, concavity, concave points, "
            "symmetry, fractal dimension)."
        ),
        "model_creator": "Sourangshu Pal (AAI-540 Group 1)",
        "model_artifact": [model_data],
        "algorithm_type": "XGBoost (gradient-boosted decision trees)",
        "model_type": "Binary classification (binary:logistic)",
        "problem_type": "Breast cancer malignancy risk scoring",
        "version": 1.0,
    },
    "intended_uses": {
        "purpose_of_model": (
            "Rank/order cytology samples by predicted malignancy probability so "
            "higher-risk samples receive specialist review first."
        ),
        "intended_users": "ML engineers and data scientists maintaining the AAI-540 lab pipeline.",
        "out_of_scope": [
            "Not a medical diagnostic device; probabilities are decision support only.",
            "Not validated for screening populations or any real clinical use.",
            "Not trained or evaluated on demographics; no fairness claims.",
        ],
    },
    "training_details": {
        "objective_function": (
            "binary:logistic (log loss); validation channel used for early-stopping-style eval."
        ),
        "training_observations": (
            "Single ml.m5.xlarge instance, File input mode, ~1 minute training time."
        ),
        "training_job": {
            "arn": info["TrainingJobArn"],
            "name": job_name,
            "training_data_source": f"s3://{bucket}/{prefix}/train (80% of WDBC, id dropped)",
            "validation_data_source": f"s3://{bucket}/{prefix}/validation (10%)",
        },
    },
    "hyperparameters": [
        {"name": "objective", "value": "binary:logistic"},
        {"name": "max_depth", "value": "5"},
        {"name": "eta", "value": "0.2"},
        {"name": "gamma", "value": "4"},
        {"name": "min_child_weight", "value": "6"},
        {"name": "subsample", "value": "0.8"},
        {"name": "num_round", "value": "100"},
        {"name": "verbosity", "value": "0"},
    ],
    "evaluation_details": [
        {
            "name": "holdout-batch",
            "evaluation_observation": (
                "10% holdout scored via Batch Transform (ml.m5.xlarge) with id joined "
                "back via input/output filters; predictions inspected for calibrated "
                "ranking against known diagnosis."
            ),
            "evaluation_job_arn": f"batch transform from training job {job_name}",
        }
    ],
    "additional_information": {
        "ethical_considerations": (
            "Educational use on a public benchmark dataset; no PHI/PII. Predictions "
            "must never be presented to patients as diagnoses."
        ),
        "caveats_and_recommendations": (
            "569-row dataset with class imbalance handled only implicitly; before any "
            "real use, add model/data quality, bias, and explainability monitoring "
            "(planned in Module 5), and require human approval (status is "
            "PendingManualApproval) before deployment."
        ),
        "deployments": [
            f"Real-time endpoint (temporary, deleted after smoke test) from endpoint config on {instance_type}",
            f"Batch transform job on {instance_type} with input_filter/join/output_filter",
        ],
    },
}

sagemaker_client.create_model_card(
    ModelCardName=MODEL_CARD_NAME,
    ModelCardStatus="Draft",
    Content=model_card_content,
)
print(f"created model card: {MODEL_CARD_NAME}")

In [ ]:
# 📸 SCREENSHOT 3 (Part 3): describe_model_card output
sagemaker_client.describe_model_card(ModelCardName=MODEL_CARD_NAME)

---
## Final submission

1. 📸 **Screenshot 1** — `describe_model_package_group` output (Part 1 cell above).
2. 📸 **Screenshot 2** — `describe_model_package` output (Part 2 cell above).
3. 📸 **Screenshot 3** — `describe_model_card` output (Part 3 cell above).
4. Insert the three screenshots under the headings in this notebook, then export the executed notebook as **PDF**:
   - Download the `.ipynb` from Studio, then locally: `jupyter nbconvert --to html Sourangshu_Pal_Assignment4.1.ipynb` → open the HTML in a browser → Print → Save as PDF (avoids the LaTeX requirement of `--to pdf`).
5. Submit **`Sourangshu_Pal_Assignment4.1.pdf`** (combined document) and **`Sourangshu_Pal_Assignment4.1.ipynb`** to Canvas.

**Cleanup notes:** the temporary endpoint is already deleted in Section 1.7. The model package group, model package, and model card are intentionally left in the Model Registry for grading.